In [56]:
%pip install torch torchvision torchaudio
%pip install seaborn
%pip install scikit-learn
%pip install tensorflow



Could not find platform independent libraries <prefix>



Note: you may need to restart the kernel to use updated packages.


Could not find platform independent libraries <prefix>


Note: you may need to restart the kernel to use updated packages.


Could not find platform independent libraries <prefix>


Note: you may need to restart the kernel to use updated packages.


Could not find platform independent libraries <prefix>


In [57]:
import pandas as pd
import torch 
from torch import nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [58]:
import pandas as pd
import csv
import numpy as np

# Load the dataset
data = pd.read_csv('Car_Kh24.csv')

df = pd.DataFrame(data)
print(df.head())

    Ad ID       Category   Locations  Car Makes     Car Model    Year  \
0  9539303  Cars for Sale  Phnom Penh    Toyota    Highlander  2003.0   
1  9529408  Cars for Sale  Phnom Penh     Lexus            NX  2015.0   
2  9540392  Cars for Sale  Phnom Penh    Toyota  Land Cruiser  2022.0   
3  9524160  Cars for Sale  Phnom Penh     Lexus         RX330  2004.0   
4  9480308  Cars for Sale  Phnom Penh     Lexus            NX  2015.0   

       Tax Type Condition Body Type    Fuel Transmission  Color  \
0  Plate Number      Used    Sports  Petrol         Auto  Black   
1     Tax Paper      Used       NaN  Petrol         Auto  White   
2  Plate Number      Used       SUV  Petrol         Auto  Black   
3  Plate Number      Used       SUV  Petrol         Auto   Gray   
4     Tax Paper      Used       NaN  Petrol         Auto  White   

                                                Link  \
0  https://www.khmer24.com/en/cars/highlander-%E1...   
1  https://www.khmer24.com/en/cars/lexus-nx200

In [59]:
# show info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17873 entries, 0 to 17872
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Ad ID         17873 non-null  int64  
 1   Category      17873 non-null  object 
 2   Locations     17873 non-null  object 
 3   Car Makes     17870 non-null  object 
 4   Car Model     17750 non-null  object 
 5   Year          17871 non-null  float64
 6   Tax Type      17873 non-null  object 
 7   Condition     17873 non-null  object 
 8   Body Type     14215 non-null  object 
 9   Fuel          15712 non-null  object 
 10  Transmission  16387 non-null  object 
 11  Color         17685 non-null  object 
 12  Link          17873 non-null  object 
 13  Title         17873 non-null  object 
 14  Price         17873 non-null  object 
 15  Year Used     17871 non-null  float64
dtypes: float64(2), int64(1), object(13)
memory usage: 2.2+ MB


In [60]:
data.isnull().sum()

Ad ID              0
Category           0
Locations          0
Car Makes          3
Car Model        123
Year               2
Tax Type           0
Condition          0
Body Type       3658
Fuel            2161
Transmission    1486
Color            188
Link               0
Title              0
Price              0
Year Used          2
dtype: int64

In [61]:
# show unique rows
df.nunique()

Ad ID           17867
Category            1
Locations          25
Car Makes          66
Car Model         515
Year               43
Tax Type            2
Condition           2
Body Type           9
Fuel                5
Transmission        2
Color              14
Link            17867
Title           15393
Price            1723
Year Used          43
dtype: int64

In [62]:
# find duplicates
duplicates = df[df.duplicated()]

# count duplicates
print(f"Number of duplicate rows: {len(duplicates)}")

#drop duplicates
df = df.drop_duplicates()


Number of duplicate rows: 6


In [63]:
print(df.columns)
# List of columns to drop
columns_to_drop = ['Ad ID', 'Link', 'Title', 'Category']

# Drop columns that exist in the DataFrame
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

# Drop rows where the columns 'Car Makes', 'Year', 'Car Model' are null
df = df.dropna(subset=['Car Makes', 'Year', 'Car Model'])


Index(['Ad ID ', 'Category ', 'Locations ', 'Car Makes', 'Car Model', 'Year',
       'Tax Type', 'Condition', 'Body Type', 'Fuel', 'Transmission', 'Color',
       'Link', 'Title', 'Price', 'Year Used'],
      dtype='object')


In [64]:
# # Drop columns 
# df = df.drop(['Ad ID', 'Link', 'Title', 'Category'], axis=1)

# # Drop rows where the column 'Car Makes' is null
# df = df.dropna(subset=['Car Makes','Year', 'Car Model'])

In [65]:
# Fill NaN in 'Body Type' by the most frequent 'Body Type' of the same 'Car Model'
df['Body Type'] = df.groupby('Car Model')['Body Type'].transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else "Unknown"))

print(df.isnull().sum())

Ad ID              0
Category           0
Locations          0
Car Makes          0
Car Model          0
Year               0
Tax Type           0
Condition          0
Body Type          0
Fuel            2129
Transmission    1454
Color            187
Price              0
Year Used          0
dtype: int64


In [66]:
# Fill NaN in 'Fuel' by the most frequent 'Fuel' type for the same 'Car Model'
df['Fuel'] = df.groupby('Car Model')['Fuel'].transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else "Unknown"))
df['Transmission'] = df.groupby('Car Model')['Transmission'].transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else "Unknown"))

print(df.isnull().sum())

Ad ID             0
Category          0
Locations         0
Car Makes         0
Car Model         0
Year              0
Tax Type          0
Condition         0
Body Type         0
Fuel              0
Transmission      0
Color           187
Price             0
Year Used         0
dtype: int64


In [67]:
# replace Color na to 'Unknown'
df['Color'] = df['Color'].fillna('Unknown')

df.isnull().sum()

Ad ID           0
Category        0
Locations       0
Car Makes       0
Car Model       0
Year            0
Tax Type        0
Condition       0
Body Type       0
Fuel            0
Transmission    0
Color           0
Price           0
Year Used       0
dtype: int64

In [68]:
# Print all column names in the DataFrame
# print(df.columns)
print(df.head())


    Ad ID       Category   Locations  Car Makes     Car Model    Year  \
0  9539303  Cars for Sale  Phnom Penh    Toyota    Highlander  2003.0   
1  9529408  Cars for Sale  Phnom Penh     Lexus            NX  2015.0   
2  9540392  Cars for Sale  Phnom Penh    Toyota  Land Cruiser  2022.0   
3  9524160  Cars for Sale  Phnom Penh     Lexus         RX330  2004.0   
4  9480308  Cars for Sale  Phnom Penh     Lexus            NX  2015.0   

       Tax Type Condition Body Type    Fuel Transmission  Color      Price  \
0  Plate Number      Used    Sports  Petrol         Auto  Black   $14,000    
1     Tax Paper      Used       SUV  Petrol         Auto  White   $39,500    
2  Plate Number      Used       SUV  Petrol         Auto  Black  $155,000    
3  Plate Number      Used       SUV  Petrol         Auto   Gray   $20,500    
4     Tax Paper      Used       SUV  Petrol         Auto  White   $51,999    

   Year Used  
0       20.0  
1        8.0  
2        1.0  
3       19.0  
4        8.0  


In [69]:
print(df['Price'])
df['Price'] = df['Price'].str.replace('$', '').str.replace(',', '').str.strip()
df['Price'] = df['Price'].astype(float)

# print(df.head())

0         $14,000 
1         $39,500 
2        $155,000 
3         $20,500 
4         $51,999 
           ...    
17868     $29,000 
17869     $18,500 
17870     $10,800 
17871     $14,000 
17872     $99,990 
Name: Price, Length: 17740, dtype: object


In [70]:
data = df.copy()
print(data.head())

    Ad ID       Category   Locations  Car Makes     Car Model    Year  \
0  9539303  Cars for Sale  Phnom Penh    Toyota    Highlander  2003.0   
1  9529408  Cars for Sale  Phnom Penh     Lexus            NX  2015.0   
2  9540392  Cars for Sale  Phnom Penh    Toyota  Land Cruiser  2022.0   
3  9524160  Cars for Sale  Phnom Penh     Lexus         RX330  2004.0   
4  9480308  Cars for Sale  Phnom Penh     Lexus            NX  2015.0   

       Tax Type Condition Body Type    Fuel Transmission  Color     Price  \
0  Plate Number      Used    Sports  Petrol         Auto  Black   14000.0   
1     Tax Paper      Used       SUV  Petrol         Auto  White   39500.0   
2  Plate Number      Used       SUV  Petrol         Auto  Black  155000.0   
3  Plate Number      Used       SUV  Petrol         Auto   Gray   20500.0   
4     Tax Paper      Used       SUV  Petrol         Auto  White   51999.0   

   Year Used  
0       20.0  
1        8.0  
2        1.0  
3       19.0  
4        8.0  


In [71]:
import pandas as pd

# Calculate Q1 (25th percentile) and Q3 (75th percentile)
Q1 = data['Price'].quantile(0.25)
Q3 = data['Price'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out rows with outliers
data_no_outliers = data[(data['Price'] >= lower_bound) & (data['Price'] <= upper_bound)]

data = data.drop(data[(data['Price'] < lower_bound) | (data['Price'] > upper_bound)].index)

# Display summary of the updated DataFrame
print(f"Original dataset size: {len(data)}")
print(f"Dataset size after removing outliers: {len(data_no_outliers)}")


Original dataset size: 16131
Dataset size after removing outliers: 16131


In [72]:
# Print the minimum and maximum values
print("Min Price:", data['Price'].min())
print("Max Price:", data['Price'].max())

# Remove rows where Price equals the minimum value (e.g., 0 or other outlier)
show = data[data['Price'] > 30000]

# Display the cleaned dataset
print(len(show))


Min Price: 500.0
Max Price: 58888.0
2779


In [73]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Preprocessing: Handle categorical variables
label_encoders = {}
print(data.head())
categorical_columns = ['Car Model', 'Tax Type', 'Fuel', 'Transmission', 'Car Makes', 'Color', 'Condition', 'Body Type']	

for col in categorical_columns:
    label_encoders[col] = LabelEncoder()
    data[col] = label_encoders[col].fit_transform(data[col])
    

print(data.head())
# Separate features and target variable
X = data[['Car Model', 'Year Used', 'Car Makes', 'Fuel', 'Transmission', 'Tax Type', 'Color', 'Condition']]  # Features
y = data['Price']  # Target variable

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train)
# Standardize the numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

    Ad ID       Category   Locations  Car Makes   Car Model    Year  \
0  9539303  Cars for Sale  Phnom Penh    Toyota  Highlander  2003.0   
1  9529408  Cars for Sale  Phnom Penh     Lexus          NX  2015.0   
3  9524160  Cars for Sale  Phnom Penh     Lexus       RX330  2004.0   
4  9480308  Cars for Sale  Phnom Penh     Lexus          NX  2015.0   
5  9252973  Cars for Sale  Phnom Penh   Hyundai       Truck  2005.0   

       Tax Type Condition Body Type    Fuel Transmission  Color    Price  \
0  Plate Number      Used    Sports  Petrol         Auto  Black  14000.0   
1     Tax Paper      Used       SUV  Petrol         Auto  White  39500.0   
3  Plate Number      Used       SUV  Petrol         Auto   Gray  20500.0   
4     Tax Paper      Used       SUV  Petrol         Auto  White  51999.0   
5  Plate Number      Used     Other  Petrol       Manual  White  14000.0   

   Year Used  
0       20.0  
1        8.0  
3       19.0  
4        8.0  
5       18.0  
    Ad ID       Category  

In [76]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Initialize the model
model = Sequential()

# Add input layer (first hidden layer)
model.add(Dense(64, input_dim=X_train.shape[1], activation='relu'))

# Add hidden layers
# model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(8, activation='relu'))
# model.add(Dense(4, activation='relu'))
# model.add(Dense(2, activation='relu'))

# Add output layer (predicting price)
model.add(Dense(1))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# Summary of the model
model.summary()


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_38 (Dense)                │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,761 (30.32 KB)

 Trainable params: 7,761 (30.32 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from sklearn.metrics import r2_score

# Compile the model with MAE as a metric for regression tasks
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# Train the model and store history
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), verbose=1)

model.save('nural_network.h5')


Epoch 1/100
404/404 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 13599768.0000 - mae: 2285.3708 - val_loss: 31192714.0000 - val_mae: 3089.9045
Epoch 2/100
404/404 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 13943519.0000 - mae: 2301.4104 - val_loss: 32177670.0000 - val_mae: 3137.3904
Epoch 3/100
404/404 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 13934317.0000 - mae: 2306.7124 - val_loss: 31144258.0000 - val_mae: 3016.4475
Epoch 4/100
404/404 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 13741161.0000 - mae: 2295.6042 - val_loss: 32758620.0000 - val_mae: 3138.7063
Epoch 5/100
404/404 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 13448971.0000 - mae: 2257.8257 - val_loss: 31253744.0000 - val_mae: 3065.3235
Epoch 6/100
404/404 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 13979293.0000 - mae: 2319.2212 - val_loss: 32695554.0000 - val_mae: 3121.1636
Epoch 7/100
404/404 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 13925901.0000 - mae: 2294.5703 - val_loss: 32301162.0000 - val_mae: 3113.1975
Epoch 8/100
404/404 ━━━━━━━

ValueError: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `model.export(filepath)` if you want to export a SavedModel for use with TFLite/TFServing/etc. Received: filepath=nural_network.m5.

In [83]:
from tensorflow.keras.models import load_model

# Load the saved model
loaded_model = load_model('nural_network.h5')

# Now you can use 'loaded_model' to make predictions or evaluate it further


In [101]:
# Test
# Predict car prices
predictions = model.predict(X_test)

# Calculate R-squared (R² score)
r2 = r2_score(y_test, predictions.flatten())
print(f"Mean Squared Error: {history.history['loss'][-1]}")
print(f"R-squared (R²) Score: {r2}")

# Show the predictions alongside the actual values
results = pd.DataFrame({'Actual': y_test, 'Predicted': predictions.flatten()})
print(len(results))
print(results.tail(10))

101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 578us/step
Mean Squared Error: 13589825.0
R-squared (R²) Score: 0.7927664096514622
3227
        Actual     Predicted
3833   13300.0  27488.437500
8235    9300.0   9780.038086
9161   10600.0  10820.118164
602     8900.0   9461.667969
16952  21999.0  47918.964844
5275   10700.0   9500.104492
14047  14900.0  15555.470703
8426   37000.0  35412.343750
8568    7699.0   9803.660156
10134  18500.0  27318.328125


In [111]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

# New car data (replace with actual car details)
new_car_data = {
    'Car Model': ['Corolla', 'Civic', 'LX570'],  # Example car model
    'Tax Type': ['Plate Number', 'Plate Number', 'Plate Number'],       # Example tax type
    'Fuel': ['Petrol', 'Diesel', 'Petrol'],        # Example fuel type
    'Transmission': ['Auto', 'Manual', 'Auto'],  # Example transmission type
    'Car Makes': ['Toyota', 'Honda', 'Lexus'],   # Example car make
    'Year Used': [20, 5.0, 2],             # Example car year
    'Color': ['Black', 'White', 'Black'],  # Example car color
    'Condition': ['New', 'Used', 'Used']  # Example car condition
}


# Convert new car data into DataFrame
new_car_df = pd.DataFrame(new_car_data)

# Handle categorical variables using the same LabelEncoder used in the training phase
label_encoders = {
    'Car Model': LabelEncoder(),
    'Tax Type': LabelEncoder(),
    'Fuel': LabelEncoder(),
    'Transmission': LabelEncoder(),
    'Car Makes': LabelEncoder(),
    'Year Used': LabelEncoder(),
    'Color': LabelEncoder(),
    'Condition': LabelEncoder()
}

# Preprocess the new car data (same encoding as done during training)
for col in new_car_data.keys():
    if col != "Year Used":
        new_car_df[col] = label_encoders[col].fit_transform(new_car_df[col])

print(new_car_df)
# Standardize the new car data (same scaling as done during training)
scaler = StandardScaler()
new_car_data_transformed = scaler.fit_transform(new_car_df)

# Assuming 'model' is your trained model, use it to predict the price of the new car
predicted_price = model.predict(new_car_data_transformed)

# Output the predicted price
for i, price in enumerate(predicted_price):
    print(f"Predicted Price for Car {i+1}: ${price[0]:,.2f}")


   Car Model  Tax Type  Fuel  Transmission  Car Makes  Year Used  Color  \
0          1         0     1             0          2       20.0      0   
1          0         0     0             1          1        5.0      1   
2          2         0     1             0          0        2.0      1   

   Condition  
0          0  
1          1  
2          1  
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Predicted Price for Car 1: $25,660.58
Predicted Price for Car 2: $21,751.38
Predicted Price for Car 3: $15,996.57


In [50]:
# Strip trailing spaces from the column names
# df.columns = df.columns.str.strip()

# # Now access the 'Car Makes' column without the KeyError
# city = df['Car Makes']
# print(city.unique())
print(df['Tax Type'].unique())


['Plate Number' 'Tax Paper']


## Ridge


In [51]:
# Ridge Regression
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score


In [52]:
# Assuming 'data' is your DataFrame
# Remove rows with missing values

data1 = df.copy()
print(data1.head())


    Ad ID       Category   Locations  Car Makes     Car Model    Year  \
0  9539303  Cars for Sale  Phnom Penh    Toyota    Highlander  2003.0   
1  9529408  Cars for Sale  Phnom Penh     Lexus            NX  2015.0   
2  9540392  Cars for Sale  Phnom Penh    Toyota  Land Cruiser  2022.0   
3  9524160  Cars for Sale  Phnom Penh     Lexus         RX330  2004.0   
4  9480308  Cars for Sale  Phnom Penh     Lexus            NX  2015.0   

       Tax Type Condition Body Type    Fuel Transmission  Color     Price  \
0  Plate Number      Used    Sports  Petrol         Auto  Black   14000.0   
1     Tax Paper      Used       SUV  Petrol         Auto  White   39500.0   
2  Plate Number      Used       SUV  Petrol         Auto  Black  155000.0   
3  Plate Number      Used       SUV  Petrol         Auto   Gray   20500.0   
4     Tax Paper      Used       SUV  Petrol         Auto  White   51999.0   

   Year Used  
0       20.0  
1        8.0  
2        1.0  
3       19.0  
4        8.0  


In [ ]:
# Assuming 'data' is your DataFrame
# Remove rows with missing values
data1 = data1.dropna()

# Select features (independent variables) and target (Price)
X = data1.drop(columns=['Price'])  # Replace with your feature columns
y = data1['Price']

# Check the data types of features
print(X.dtypes)
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
for column in X.select_dtypes(include='object').columns:
    X[column] = label_encoder.fit_transform(X[column])





Ad ID             int64
Category         object
Locations        object
Car Makes        object
Car Model        object
Year            float64
Tax Type         object
Condition        object
Body Type        object
Fuel             object
Transmission     object
Color            object
Year Used       float64
dtype: object


In [54]:
# Re-split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [55]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'max_iter': [500, 1000, 2000],
    'eta0': [0.01, 0.1, 0.5],  # Initial learning rate
    'learning_rate': ['constant', 'optimal', 'invscaling', 'adaptive']
}

# Initialize SGDRegressor
sgd = SGDRegressor(penalty='l2', random_state=42)

# Grid search
grid_search = GridSearchCV(sgd, param_grid, scoring='neg_mean_squared_error', cv=5)
grid_search.fit(X_train_scaled, y_train)

# Best parameters
print("Best Parameters:", grid_search.best_params_)

# Train with best parameters
sgd_model = grid_search.best_estimator_


NameError: name 'SGDRegressor' is not defined

In [81]:
# Capping outliers in Price
upper_limit = y_train.quantile(0.99)  # Top 1% cap
lower_limit = y_train.quantile(0.01)  # Bottom 1% cap

y_train = y_train.clip(lower=lower_limit, upper=upper_limit)

import numpy as np

# Apply log transformation to Price
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# Predict and evaluate in the transformed space
sgd_model.fit(X_train_scaled, y_train_log)
y_pred_log = sgd_model.predict(X_test_scaled)

# Inverse transform predictions for evaluation
y_pred = np.expm1(y_pred_log)



In [82]:
sgd_model = SGDRegressor(
    penalty='l2',
    alpha=0.01,
    max_iter=2000,
    eta0=0.01,  # Initial learning rate
    learning_rate='adaptive',  # Dynamically adjusts learning rate
    random_state=42
)
sgd_model.fit(X_train_scaled, y_train)


SGDRegressor(alpha=0.01, learning_rate='adaptive', max_iter=2000,
             random_state=42)

In [83]:
sgd_model = SGDRegressor(
    penalty='l2',
    alpha=0.01,
    max_iter=5000,  # Increase maximum iterations
    tol=1e-4,  # Convergence tolerance
    random_state=42
)
sgd_model.fit(X_train_scaled, y_train)


SGDRegressor(alpha=0.01, max_iter=5000, random_state=42, tol=0.0001)

In [84]:
from sklearn.model_selection import cross_val_score

# Evaluate with cross-validation
cv_scores = cross_val_score(sgd_model, X_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')
print("Average CV MSE:", -np.mean(cv_scores))


Average CV MSE: 450523040.77168477


In [86]:
%pip install xgboost


   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/124.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.9 MB 932.9 kB/s eta 0:02:14
   ---------------------------------------- 0.8/124.9 MB 1.2 MB/s eta 0:01:48
   ---------------------------------------- 1.0/124.9 MB 1.3 MB/s eta 0:01:36
    --------------------------------------- 1.8/124.9 MB 1.7 MB/s eta 0:01:13
    --------------------------------------- 2.4/124.9 MB 1.9 MB/s eta 0:01:05
    --------------------------------------- 2.9/124.9 MB 2.0 MB/s eta 0:01:02
   - -------------------------------------- 3.1/124.9 MB 1.9 MB/s eta 0:01:04
   - -------------------------------------- 3.4/124.9 MB 1.9 MB/s eta 0:01:04
   - -------------------------------------- 3.7/124.9 MB 1.8 MB/s eta 0:01:06
   - -------------------------------------- 3.9/124.9 MB 1.8 MB/s eta 0:01:08
   - 

Could not find platform independent libraries <prefix>


In [85]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost MSE:", mean_squared_error(y_test, y_pred_xgb))
print("XGBoost R² Score:", r2_score(y_test, y_pred_xgb))


ModuleNotFoundError: No module named 'xgboost'

In [72]:
# Calculate evaluation metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Print results
print("Mean Squared Error (MSE):", mse)
print("R^2 Score:", r2)


Mean Squared Error (MSE): 1025465639.1663109
R^2 Score: 0.29847499559972157


In [73]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid for alpha
param_grid = {'alpha': [0.1, 1.0, 10.0, 100.0]}

# Initialize the Ridge model
ridge = Ridge()

# Perform grid search
grid_search = GridSearchCV(ridge, param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)

# Best alpha value
print("Best Alpha:", grid_search.best_params_['alpha'])


Best Alpha: 10.0


In [74]:
# Train Ridge with optimal alpha
best_alpha = grid_search.best_params_['alpha']
ridge_model_optimized = Ridge(alpha=best_alpha)
ridge_model_optimized.fit(X_train_scaled, y_train)

# Predict and evaluate
y_pred_optimized = ridge_model_optimized.predict(X_test_scaled)
print("Optimized MSE:", mean_squared_error(y_test, y_pred_optimized))
print("Optimized R^2 Score:", r2_score(y_test, y_pred_optimized))


Optimized MSE: 1025485732.850342
Optimized R^2 Score: 0.29846124943286834


In [26]:
# Car_Makes = ['Toyota', 'Lexus', 'Hyundai', 'Land Rover', 'Jaguar', 'Mercedes-Benz', 'Jeep',
#  'Kia', 'Mitsubishi', 'Ford', 'Maserati', 'Nissan', 'Peugeot', 'CHANGAN', 'Mazda',
#  'Honda', 'Audi', 'SsangYong', 'Dodge', 'Volkswagen', 'Porsche', 'Tesla', 'BMW',
#  'Isuzu', 'MG', 'Chevrolet', 'Daewoo', 'Infiniti', 'HUMMER', 'Mini',
#  'Rolls-Royce', 'Lamborghini', 'Aston Martin', 'Cadillac', 'Bentley', 'McLaren',
#  'ZX AUTO', 'Renault', 'BAIC', 'Subaru', 'Suzuki', 'HONGQI', 'Smart', 'WULING',
#  'MAXUS', 'Volvo', 'Ferrari', 'Chrysler', 'Daihatsu', 'GAC', 'Alfa Romeo', 'Ram',
#  'Foton', 'Haval', 'Great Wall', 'FORTHING', 'Changhe', 'SOUEAST', 'BESTUNE',
#  'Hawtai Motor', 'JETOUR', 'Acura', 'ZOTYE', 'BYD', 'JAC']

# # Print the list of car makes
# print(Car_Makes)


# # replace 'Car Makes' with the index of the car make in the list
# df['Car Makes'] = df['Car Makes'].apply(lambda x: Car_Makes.index(x))
# df.head()

In [27]:
# Strip trailing spaces from the column names
df.columns = df.columns.str.strip()

# Access the 'Car Model' column and get unique values as a list
city = df['Car Model'].unique().tolist()

# Print the list
print(city)


['Highlander', 'NX', 'Land Cruiser', 'RX330', 'Truck', 'Vitz', 'Range Rover Evoque', 'Prius', 'LX470', 'Belta', 'Alphard', 'F-Type', 'S350', 'Wrangler', 'S400L', 'CT', 'Ray', 'Montero', 'E-Class', 'Ranger Wildtrak', 'RX400h', 'RAV4', 'Land Cruiser PRADO', 'Range Rover Vogue', 'Camry', 'Corolla Altis', 'Triton', 'RX300', 'H350', 'Ghibli iii S', 'Morning', 'Terra', 'LX450d', 'Territory', '3008', 'RC', 'Range Rover', 'UNI-T', 'CX-9', 'C300', 'Rubicon', 'RX450h', 'Cx-8', 'CR-V', 'A8', '5008', 'CS35 PLUS', '4Runner', 'UNI-K', 'GS', 'Hilux REVO', 'F-150 Raptor', 'Mazda3', 'Mustang', 'Cx-3', 'BT50', 'Ecosport', 'Istana - ធូរីស', 'Mazda6', 'Matrix', 'Scion', 'Tivoli', 'RX350', 'GX', 'Corolla', 'Visto', 'RX200T', 'Tacoma', 'Frontier', 'Challenger', 'Range Rover Sport', 'Hiace', 'Atlas', 'Cube', 'Boxster', 'Navara', 'BT50 Pro', 'Panamera', 'Staria', 'Tundra', 'Q5', 'Raize', 'Model Y', 'Model 3', 'Q7', 'Starex', 'Q8', 'Macan S', '740LI', 'LX570', 'LM', 'X5', 'D-MAX', 'Hilux Vigo', 'Trooper', 'M3'

In [28]:
# Strip trailing spaces from the column names
df.columns = df.columns.str.strip()

# Access the 'Car Model' column and get unique values as a list
tax_type = df['Tax Type'].unique().tolist()

# Print the list
print(tax_type)

# replace 'Tax Type' with the index of the tax type in the list

# 0 for Plate Number Tax and 1 for Tax Paper
# df['Tax Type'] = df['Tax Type'].apply(lambda x: tax_type.index(x))
# df.head()


['Plate Number', 'Tax Paper']


In [29]:
# Strip trailing spaces from the column names
df.columns = df.columns.str.strip()

# Access the 'Car Model' column and get unique values as a list
tax_type = df['Tax Type'].unique().tolist()

# Print the list
print(tax_type)

# replace 'Tax Type' with the index of the tax type in the list

# # 0 for Plate Number Tax and 1 for Tax Paper
# df['Tax Type'] = df['Tax Type'].apply(lambda x: tax_type.index(x))
# df.head()


['Plate Number', 'Tax Paper']


In [30]:
# Strip trailing spaces from the column names
df.columns = df.columns.str.strip()

# Access the 'Condition' column and get unique values as a list
tax_type = df['Condition'].unique().tolist()

# Print the list
print(tax_type)

# # replace 'Condition' with the index of the Condition in the list
"""
0 for Used
1 for New
"""

# df['Condition'] = df['Condition'].apply(lambda x: tax_type.index(x))
df.head()


['Used', 'New']


,Ad ID,Category,Locations,Car Makes,Car Model,Year,Tax Type,Condition,Body Type,Fuel,Transmission,Color,Price,Year Used
0,9539303,Cars for Sale,Phnom Penh,Toyota,Highlander,2003.0,Plate Number,Used,Sports,Petrol,Auto,Black,14000.0,20.0
1,9529408,Cars for Sale,Phnom Penh,Lexus,NX,2015.0,Tax Paper,Used,SUV,Petrol,Auto,White,39500.0,8.0
2,9540392,Cars for Sale,Phnom Penh,Toyota,Land Cruiser,2022.0,Plate Number,Used,SUV,Petrol,Auto,Black,155000.0,1.0
3,9524160,Cars for Sale,Phnom Penh,Lexus,RX330,2004.0,Plate Number,Used,SUV,Petrol,Auto,Gray,20500.0,19.0
4,9480308,Cars for Sale,Phnom Penh,Lexus,NX,2015.0,Tax Paper,Used,SUV,Petrol,Auto,White,51999.0,8.0


In [31]:
# # Compute the correlation matrix
# correlation_matrix = df.corr()

# # Display the correlation matrix
# print(correlation_matrix)

In [32]:
# df_encoded = pd.get_dummies(df, columns=['Car Model', 'Fuel', 'Transmission'])
# print(df_encoded.head())